In [ ]:
%%sql -r dataframe_12
USE ROLE CHEETAH_DATA5035_ROLE

# MOAB Rover Survey Lab: Feature Engineering in Snowflake

## Goal
In this lab, we will engineer new features from rover-collected survey data using:
- Python UDFs
- SQL views
- AI prompts

## Raw Inputs
- Easting
- Northing
- Sensor measurement

## Engineered Features
1. Grid tile, survey unit, and subcell assignment
2. Tile-level aggregated measurement signals
3. Comparison to normal range
4. Remediation prioritization

## SCALARS Mapping
- **Simplify**: convert coordinates into tile IDs
- **Aggregate**: summarize measurements at the tile level using multi-level aggregation
- **Assess**: compare readings to expected range
- **Rank / Score**: prioritize areas for remediation

In [ ]:
%%sql -r source_data
-- Change to your user's schema
USE SCHEMA data5035.CHEETAH;
SELECT * FROM data5035.spring26.sdg_001_ra226_scandata limit 10;

## Convert from Coordinates to Grid

The input data is provided in directional distances on a flat map projection. Northing and Easting indicate how far to go in those directions (up and right) in US Feet relative to a known starting point. Our purpose here is to map those directional distances on to a three-level grid.

### Measurement
* **Tiles** are the largest areas. They are composed of a grid of **Survey Units** 21 tiles wide x 18 tiles tall.
* **Survey Units** measure 32.81 ft x 32.81 ft square
* **Subcells** are square subdivisions within the **Survey Units** laid out 10x10

### Labeling
* **Tiles** are coded by row letter and column letter starting at AA in the bottom-left of our map, given some known origin (2180160.001, 6660000.000). AA indicates 1st row, 1st column. AB indicates 1st row, 2nd column to the left. BA indicates 2nd row up, 1st column.
* Within each Tile, **Survey Units** are numbered starting in the top-left corner, proceeeding right, then down to the beginning of the next row (as if you're reading down a page)
* Within each Survey Unit, **Subcells** are numbered starting in the bottom-left corner, proceeduing right, then up to the beginning of thext row (as if you're reading from the bottom of a page up)

**Create a Python UDF to convert x (easting), y (northing) into the grid labels.**

### Testing

```
    >>> convert_xy(2180160.0001, 6660000.0000)  
    ('AA', 358, 1)

    >>> convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000)
    ('AB', 358, 1)

    >>> convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4)
    ('BB', 338, 12)
```

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE FUNCTION CONVERT_XY(
        X FLOAT,
        Y FLOAT,
        ORIGIN_X FLOAT,
        ORIGIN_Y FLOAT,
        SU_SIZE FLOAT,
        TILE_GRID_X NUMBER(38,0),
        TILE_GRID_Y NUMBER(38,0),
        SUBCELL_GRID NUMBER(38,0)
    )
    RETURNS OBJECT
    LANGUAGE PYTHON
    RUNTIME_VERSION = '3.11'
    HANDLER = 'convert_xy'
    AS 
    $$
def convert_xy(x, y, origin_x, origin_y, su_size, tile_grid_x, tile_grid_y, subcell_grid):

    # Offset from origin (bottom-left of tile AA)
    dx = x - origin_x
    dy = y - origin_y

    if dx < 0 or dy < 0:
        return {"tile": "BAD_DATA", "survey_unit": None, "sub_cell": None,
                "error": f"Negative offset: dx={dx:.4f}, dy={dy:.4f}"}

    # Tile indices (0-indexed: 0=A, 1=B, ...)
    tile_width  = su_size * tile_grid_x
    tile_height = su_size * tile_grid_y

    tile_col_idx = int(dx / tile_width)
    tile_row_idx = int(dy / tile_height)

    if tile_col_idx > 25 or tile_row_idx > 25:
        return {"tile": "BAD_DATA", "survey_unit": None, "sub_cell": None,
                "error": f"Tile index exceeds alphabet (col={tile_col_idx}, row={tile_row_idx})"}

    # First letter = row (northing), second = column (easting)
    tile_code = chr(ord('A') + tile_row_idx) + chr(ord('A') + tile_col_idx)

    # Position within the tile
    x_in_tile = dx - tile_col_idx * tile_width
    y_in_tile = dy - tile_row_idx * tile_height

    # Survey Unit: numbered top-left=1, right then down
    su_col          = min(int(x_in_tile / su_size), tile_grid_x - 1)
    su_row_from_bot = min(int(y_in_tile / su_size), tile_grid_y - 1)
    su_row_from_top = (tile_grid_y - 1) - su_row_from_bot
    su_number       = su_row_from_top * tile_grid_x + su_col + 1

    # Position within the Survey Unit
    x_in_su = x_in_tile - su_col          * su_size
    y_in_su = y_in_tile - su_row_from_bot * su_size

    # Sub-cell: numbered bottom-left=1, right then up
    sc_size         = su_size / subcell_grid
    sc_col          = min(int(x_in_su / sc_size), subcell_grid - 1)
    sc_row_from_bot = min(int(y_in_su / sc_size), subcell_grid - 1)
    sc_number       = sc_row_from_bot * subcell_grid + sc_col + 1

    return {"tile": tile_code, "survey_unit": su_number, "sub_cell": sc_number}
    $$;

In [ ]:
%%sql -r dataframe_3
    CREATE OR REPLACE FUNCTION CONVERT_XY(X FLOAT, Y FLOAT)
    RETURNS OBJECT
    LANGUAGE SQL
    AS
    $$
        SELECT CONVERT_XY(
            X, Y,
            2180160.0001,
            6660000.0000,
            32.81,
            21,
            18,
            10
        )
    $$;

In [ ]:
%%sql -r dataframe_4
select convert_xy(2180160.0001, 6660000.0000);

In [ ]:
%%sql -r dataframe_5
select convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000);

In [ ]:
%%sql -r dataframe_6
select convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4)

In [ ]:
%%sql -r dataframe_7
SELECT 
    convert_xy(easting, northing) AS coordinates, 
    coordinates:su::INTEGER AS su,
    coordinates:subcell::INTEGER AS subcell,
    coordinates:tile::STRING AS tile,
    * 
FROM 
    data5035.spring26.sdg_001_ra226_scandata 
LIMIT 100;

## Compute Layered Averages

*Creates an equal-weight average where each level treats its child level equally (weight is 1 per child), regardless of collected samples.*

* **Layer 1 (Subcell)**: average of all raw readings within a subcell
* **Layer 2 (Survey Unit)**: average of subcell averages
* **Layer 3 (Tile)**: average of survey unit ranges

In [ ]:
%%sql -r dataframe_8
SELECT 
    convert_xy(easting, northing) AS coordinates, 
    coordinates:tile::STRING      AS tile,
    coordinates:survey_unit::INT  AS survey_unit,
    coordinates:sub_cell::INT     AS sub_cell,
    avg(reading) 
FROM 
    data5035.spring26.sdg_001_ra226_scandata 
GROUP BY ALL
ORDER BY 2, 3, 4

In [ ]:
%%sql -r dataframe_1
-- =============================================================================
-- LAYERED AVERAGES  —  Radiation Site Survey
-- Each level treats its children equally (weight = 1 per child),
-- regardless of how many raw samples were collected there.
-- =============================================================================

WITH

-- Layer 0: attach grid labels to every raw reading
labeled AS (
    SELECT
        r.READING,
        loc:tile::VARCHAR    AS TILE,
        loc:survey_unit::INT AS SURVEY_UNIT,
        loc:sub_cell::INT    AS SUB_CELL
    FROM DATA5035.SPRING26.SDG_001_RA226_SCANDATA r,
         LATERAL (SELECT CONVERT_XY(r.EASTING, r.NORTHING) AS loc) l
    WHERE loc:tile::VARCHAR <> 'BAD_DATA'
),

-- Layer 1: one average per sub-cell
--   N raw samples in the same square meter → treated as a single value
subcell_avgs AS (
    SELECT
        TILE,
        SURVEY_UNIT,
        SUB_CELL,
        AVG(READING) AS AVG_READING
    FROM labeled
    GROUP BY TILE, SURVEY_UNIT, SUB_CELL
),

-- Layer 2: one average per survey unit  (average of sub-cell averages)
--   Each sub-cell gets weight 1 regardless of its sample count
su_avgs AS (
    SELECT
        TILE,
        SURVEY_UNIT,
        AVG(AVG_READING) AS AVG_READING
    FROM subcell_avgs
    GROUP BY TILE, SURVEY_UNIT
),

-- Layer 3: one average per tile  (average of survey unit averages)
--   Each survey unit gets weight 1 regardless of its sub-cell count
tile_avgs AS (
    SELECT
        TILE,
        AVG(AVG_READING) AS AVG_READING
    FROM su_avgs
    GROUP BY TILE
)

-- Final output — pull whichever level(s) you need:
SELECT * FROM tile_avgs       ORDER BY TILE

In [ ]:
%%sql -r dataframe_11
-- Final output — all three levels side by side
WITH
labeled AS (
    SELECT
        r.READING,
        loc:tile::VARCHAR    AS TILE,
        loc:survey_unit::INT AS SURVEY_UNIT,
        loc:sub_cell::INT    AS SUB_CELL
    FROM DATA5035.SPRING26.SDG_001_RA226_SCANDATA r,
         LATERAL (SELECT CONVERT_XY(r.EASTING, r.NORTHING) AS loc) l
    WHERE loc:tile::VARCHAR <> 'BAD_DATA'
),
subcell_avgs AS (
    SELECT
        TILE,
        SURVEY_UNIT,
        SUB_CELL,
        AVG(READING) AS AVG_READING
    FROM labeled
    GROUP BY TILE, SURVEY_UNIT, SUB_CELL
),
su_avgs AS (
    SELECT
        TILE,
        SURVEY_UNIT,
        AVG(AVG_READING) AS AVG_READING
    FROM subcell_avgs
    GROUP BY TILE, SURVEY_UNIT
),
tile_avgs AS (
    SELECT
        TILE,
        AVG(AVG_READING) AS AVG_READING
    FROM su_avgs
    GROUP BY TILE
)
SELECT
    s.TILE,
    s.SURVEY_UNIT,
    s.SUB_CELL,
    ROUND(s.AVG_READING,  4) AS AVG_SUBCELL,
    ROUND(su.AVG_READING, 4) AS AVG_SU,
    ROUND(t.AVG_READING,  4) AS AVG_TILE
FROM subcell_avgs s
JOIN su_avgs  su ON s.TILE = su.TILE AND s.SURVEY_UNIT = su.SURVEY_UNIT
JOIN tile_avgs t ON s.TILE = t.TILE
ORDER BY s.TILE, s.SURVEY_UNIT, s.SUB_CELL

## Compare to Reference Ranges

This measurement is of Radium-226 levels.
* `<5` - OK
* `5 <= X < 7.4` - Warning
* `>= 7.4` - Alarm

In [ ]:
%%sql -r dataframe_9
WITH
-- CTE 1: labeled
-- Calls CONVERT_XY on every row of raw data to attach grid coordinates.
-- LATERAL calls the UDF once per row and references its result (loc)
-- multiple times in the same SELECT without repeating the call.
-- Filter out any rows that the UDF has flagged as BAD_DATA (i.e., points
-- falling outside of a grid origin, negative numbers) to drop them from the calculation.
labeled AS (
    SELECT
        r.READING,
        loc: tile:: VARCHAR AS TILE,
        loc: survey_unit::INT AS SURVEY_UNIT,
        loc: sub_cell::INT AS SUB_CELL
    FROM DATA5035.SPRING26.SDG_001_RA226_SCANDATA r,
        LATERAL (SELECT CONVERT_XY(r.EASTING, r.NORTHING) AS loc) l
    WHERE loc:tile::VARCHAR <> 'BAD_DATA'
),

-- CTE 2: subcell_avgs
-- Collapse all raw readings that share the same tile, survey unit, and subcell into a single average
-- Multiple rover passes over the subcell contribute equally to this average before rolling up.
subcell_avgs AS (
    SELECT
        TILE, SURVEY_UNIT, SUB_CELL,
        AVG(READING) AS AVG_SUBCELL
    FROM labeled
    GROUP BY TILE, SURVEY_UNIT, SUB_CELL
),

--CTE 3: su_avgs
-- Average the subcell averages to get one value per survey unit.
-- Due to averaging values that have already been averaged, each subcell 
-- now gets an equal weight, regardless of the raw samples it contains.
su_avgs AS (
    SELECT
        TILE, SURVEY_UNIT,
        AVG(AVG_SUBCELL) AS AVG_SU
    FROM subcell_avgs
    GROUP BY TILE, SURVEY_UNIT
),

--CTE 4: tile_avgs
-- Average the survey unit averages to get one value per tile.
-- Same goal as su_avgs: every survey unit gets one count, no matter size.
tile_avgs AS (
    SELECT
        TILE,
        AVG(AVG_SU) AS AVG_TILE
    FROM su_avgs
    GROUP BY TILE
)

--Final SELECT statement joins all three CTEs together so that 
-- every output row shows the subcell, its parent survey unit, and that
-- survey unit's tile. Each level has its own CASE expression that maps
-- the numeric average onto the three tier status label using the radiation thresholds
-- ROUND keeps some precision but improves readability
SELECT
    s.TILE,
    s.SURVEY_UNIT,
    s.SUB_CELL,

    -- Subcell average and its status (for ~ 3 meters squared per survey unit)
    ROUND(s.AVG_SUBCELL, 4) AS AVG_SUBCELL,
    CASE    
        WHEN s.AVG_SUBCELL >=7.4 THEN 'ALARM'
        WHEN s.AVG_SUBCELL >=5.0 THEN 'WARNING'
        ELSE 'OK'
    END AS SUBCELL_STATUS,

    -- Survey unit average and its status (~10x10 meters, contains 100 subcells)
    ROUND(su.AVG_SU, 4) AS AVG_SU,
    CASE    
        WHEN su.AVG_SU >=7.4 THEN 'ALARM'
        WHEN su.AVG_SU >=5.0 THEN 'WARNING'
        ELSE 'OK'
    END AS SU_STATUS,

    -- Tile average and its status (21x18 grid of survey units)
    ROUND(t.AVG_TILE, 4) AS AVG_TILE,
    CASE    
        WHEN t.AVG_TILE >=7.4 THEN 'ALARM'
        WHEN t.AVG_TILE >=5.0 THEN 'WARNING'
        ELSE 'OK'
    END AS TILE_STATUS

-- Join CTES on shared keys so each subcell row has its parent & grandparent average
FROM subcell_avgs s
JOIN su_avgs su ON s.TILE = su.TILE AND s.SURVEY_UNIT = su.SURVEY_UNIT
JOIN tile_avgs t on s.TILE = t.TILE

-- Sort by urgency descending-- Use CASE TO convert 
-- to integers and ordering numerically ascending.
ORDER BY
    CASE TILE_STATUS   WHEN 'ALARM' THEN 1 WHEN 'WARNING' THEN 2 ELSE 3 END,
    CASE SU_STATUS   WHEN 'ALARM' THEN 1 WHEN 'WARNING' THEN 2 ELSE 3 END,
    CASE SUBCELL_STATUS   WHEN 'ALARM' THEN 1 WHEN 'WARNING' THEN 2 ELSE 3 END,
    s.TILE, s.SURVEY_UNIT, s.SUB_CELL

---

## Exercise 12 — Additional Engineered Features

| # | Feature | Category | Description |
|---|---------|----------|-------------|
| 1 | **Weighted Average Radiation Level** | Aggregate | Raw-sample count drives the weight, so denser areas pull the average more than sparse ones |
| 2 | **Hotspot Flag** | Simplify | Binary 1/0 per tile — 1 when MAX reading in that tile >= 7.4 (ALARM threshold) |
| 3 | **Sensor Variability Index** | Assess | STDDEV / AVG per tile — coefficient of variation; higher = noisier / less trustworthy readings |

### Feature 1: Weighted Average Radiation Level

In the lab we treat subcells equally regardless of their raw sample count. This weighted average lets areas more densely sampled influence the mean proportionally.

* **Subcell weighted average**: same as regular AVG (all raw readings from cell)
* **Survey Unit weighted average**: each subcell's contribution weighted by raw sample count
* **Tile weighted average**: each survey unit contribution is weighted by its total raw sample count

In [ ]:
-- Using same approach as above for applying XY coordinates and 
-- filtering out bad data in labeled CTE)
WITH
labeled AS (
    SELECT
        r.READING,
        loc: tile:: VARCHAR AS TILE,
        loc: survey_unit::INT AS SURVEY_UNIT,
        loc: sub_cell::INT AS SUB_CELL
    FROM DATA5035.SPRING26.SDG_001_RA226_SCANDATA r,
        LATERAL (SELECT CONVERT_XY(r.EASTING, r.NORTHING) AS loc) l
    WHERE loc:tile::VARCHAR <> 'BAD_DATA'
),

-- Subcell Level: average & count of raw samples
subcell_weighted AS (
    SELECT
        TILE,
        SURVEY_UNIT,
        SUB_CELL,
        AVG(READING) AS WAVG_SUBCELL,
        COUNT(READING) AS N_SAMPLES
    FROM labeled
    GROUP BY TILE, SURVEY_UNIT, SUB_CELL
),

-- Survey unit level: weighted by each subcell's raw sample count
-- SUM(value * weight) / SUM(weight) where weight = N_SAMPLES
su_weighted AS (
    SELECT
        TILE,
        SURVEY_UNIT,
        SUM(WAVG_SUBCELL * N_SAMPLES) / NULLIF(SUM(N_SAMPLES), 0) AS WAVG_SU,
        SUM(N_SAMPLES) AS N_SAMPLES
    FROM subcell_weighted
    GROUP BY TILE, SURVEY_UNIT
),

-- Tile level: weighted by each survey unit's raw sample count
tile_weighted AS(
    SELECT
        TILE,
        SUM(WAVG_SU * N_SAMPLES) / NULLIF(SUM(N_SAMPLES), 0) AS WAVG_TILE,
        SUM(N_SAMPLES) AS N_SAMPLES
    FROM su_weighted
    GROUP BY TILE
)

-- Show all three weighted levels together (similar to comprehensive join above)
SELECT
    sc.TILE,
    sc.SURVEY_UNIT,
    sc.SUB_CELL,
    sc.N_SAMPLES AS RAW_SAMPLE_COUNT,
    ROUND(sc.WAVG_SUBCELL, 4) AS WAVG_SUBCELL,
    ROUND(su.WAVG_SU, 4) AS WAVG_SU,
    ROUND(t.WAVG_TILE, 4) AS WAVG_TILE
FROM subcell_weighted sc
JOIN su_weighted su ON sc.TILE = su.TILE AND sc.SURVEY_UNIT = su.SURVEY_UNIT
JOIN tile_weighted t ON sc.TILE = t.TILE
ORDER BY sc.TILE, sc.SURVEY_UNIT, sc.SUB_CELL;

### Feature 2: Hotspot Flag

Binary indicator of whether raw readings reach/exceed the 'ALARM' radiation threshold (7.4). Can be used as a map overlay to quickly identify any dangerous readings within a tile, even if its average is overall safe.

In [ ]:
-- Using same approach as above for applying XY coordinates and 
-- filtering out bad data in labeled CTE)
WITH
labeled AS (
    SELECT
        r.READING,
        loc: tile:: VARCHAR AS TILE,
        loc: survey_unit::INT AS SURVEY_UNIT,
        loc: sub_cell::INT AS SUB_CELL
    FROM DATA5035.SPRING26.SDG_001_RA226_SCANDATA r,
        LATERAL (SELECT CONVERT_XY(r.EASTING, r.NORTHING) AS loc) l
    WHERE loc:tile::VARCHAR <> 'BAD_DATA'
),

tile_stats AS (
    SELECT
        TILE,
        MAX(READING) AS MAX_READING,
        AVG(READING) AS AVG_READING,
        COUNT(READING) AS TOTAL_SAMPLES
    FROM labeled
    GROUP BY TILE
)

SELECT
    TILE, 
    ROUND(AVG_READING, 4) AS AVG_READING,
    ROUND(MAX_READING, 4) AS MAX_READING,
    TOTAL_SAMPLES,
    -- Hotspot flag: if max reading in tile >= ALARM threshold
    IFF(MAX_READING >=7.4, 1, 0) AS HOTSPOT_FLAG,
    -- Hotspot label to help with readability/reporting
    IFF(MAX_READING >=7.4, 'HOTSPOT', 'NORMAL') AS HOTSPOT_LABEL
FROM tile_stats
ORDER BY HOTSPOT_FLAG DESC, MAX_READING DESC;
    

### Feature 3: Sensor Variability Index

The coefficient of variation(CV) normalizes the standard deviation by the mean, allowing us to measure variability across different average radiation levels

* **Low CV** (close to 0) -> readings are tightly clustered, high confidence in average
* **High CV** -> wide spread of readings, average may be masking localized spikes

*NULLIF helps mitigate potential issues with dividing by zero where tiles have a zero mean*

In [ ]:
-- Using same approach as above for applying XY coordinates and 
-- filtering out bad data in labeled CTE)
WITH
labeled AS (
    SELECT
        r.READING,
        loc: tile:: VARCHAR AS TILE,
        loc: survey_unit::INT AS SURVEY_UNIT,
        loc: sub_cell::INT AS SUB_CELL
    FROM DATA5035.SPRING26.SDG_001_RA226_SCANDATA r,
        LATERAL (SELECT CONVERT_XY(r.EASTING, r.NORTHING) AS loc) l
    WHERE loc:tile::VARCHAR <> 'BAD_DATA'
),

tile_stats AS (
    SELECT
        TILE,
        COUNT(READING) AS N_SAMPLES,
        ROUND(AVG(READING), 4) AS AVG_READING,
        ROUND(STDDEV(READING), 4) AS STDDEV_READING,
        ROUND(
            STDDEV(READING) / NULLIF(AVG(READING), 0), 4
        ) AS VARIABILITY_INDEX
    FROM labeled
    GROUP BY TILE
)

SELECT
    TILE,
    N_SAMPLES,
    AVG_READING,
    STDDEV_READING,
    VARIABILITY_INDEX,
    -- Bucketing for easy interpretation
    CASE
        WHEN VARIABILITY_INDEX < 0.10 THEN 'LOW - stable readings'
        WHEN VARIABILITY_INDEX < 0.25 THEN 'MODERATE - some spread'
        ELSE 'HIGH - inconsistent readings'
    END AS VARIABILITY_CATEGORY
FROM tile_stats
ORDER BY VARIABILITY_INDEX DESC;

## Prioritize

Once we've build everything out, let's use AI to make some recommendations on remediation priorities.

In [ ]:
%%sql -r dataframe_10
-- =============================================================================
-- AI REMEDIATION PRIORITIZATION
-- Combines all engineered features into a single per-tile summary and passes
-- it to Snowflake Cortex COMPLETE for a natural-language priority recommendation.
-- =============================================================================

WITH
labeled AS (
    SELECT
        r.READING,
        loc:tile::VARCHAR    AS TILE,
        loc:survey_unit::INT AS SURVEY_UNIT,
        loc:sub_cell::INT    AS SUB_CELL
    FROM DATA5035.SPRING26.SDG_001_RA226_SCANDATA r,
         LATERAL (SELECT CONVERT_XY(r.EASTING, r.NORTHING) AS loc) l
    WHERE loc:tile::VARCHAR <> 'BAD_DATA'
),
subcell_avgs AS (
    SELECT TILE, SURVEY_UNIT, SUB_CELL, AVG(READING) AS AVG_SUBCELL
    FROM labeled
    GROUP BY TILE, SURVEY_UNIT, SUB_CELL
),
su_avgs AS (
    SELECT TILE, SURVEY_UNIT, AVG(AVG_SUBCELL) AS AVG_SU
    FROM subcell_avgs
    GROUP BY TILE, SURVEY_UNIT
),
tile_avgs AS (
    SELECT TILE, AVG(AVG_SU) AS AVG_TILE
    FROM su_avgs
    GROUP BY TILE
),
tile_stats AS (
    SELECT
        TILE,
        MAX(READING)                                                     AS MAX_READING,
        COUNT(READING)                                                   AS N_SAMPLES,
        ROUND(STDDEV(READING) / NULLIF(AVG(READING), 0), 4)             AS VARIABILITY_INDEX
    FROM labeled
    GROUP BY TILE
),
tile_summary AS (
    SELECT
        ta.TILE,
        ROUND(ta.AVG_TILE,       4)                                AS AVG_TILE,
        ROUND(ts.MAX_READING,    4)                                AS MAX_READING,
        ts.N_SAMPLES,
        ts.VARIABILITY_INDEX,
        IFF(ts.MAX_READING >= 7.4, 1, 0)                           AS HOTSPOT_FLAG,
        CASE
            WHEN ta.AVG_TILE >= 7.4 THEN 'ALARM'
            WHEN ta.AVG_TILE >= 5.0 THEN 'WARNING'
            ELSE 'OK'
        END                                                        AS TILE_STATUS
    FROM tile_avgs ta
    JOIN tile_stats ts ON ta.TILE = ts.TILE
)

SELECT
    TILE,
    AVG_TILE,
    MAX_READING,
    TILE_STATUS,
    HOTSPOT_FLAG,
    VARIABILITY_INDEX,

    -- AI recommendation via Snowflake Cortex
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT(
            'You are a radiation safety analyst. Given the following Ra-226 survey data for a single grid tile, ',
            'provide a concise 1-2 sentence remediation priority recommendation. ',
            'Be specific about urgency and suggested next steps.',
            CHR(10),
            'Tile: ',          TILE,
            ' | Status: ',     TILE_STATUS,
            ' | Avg reading: ', AVG_TILE::VARCHAR,
            ' | Max reading: ', MAX_READING::VARCHAR,
            ' | Hotspot flag: ', HOTSPOT_FLAG::VARCHAR,  ' (1=hotspot, 0=normal)',
            ' | Variability index: ', VARIABILITY_INDEX::VARCHAR, ' (higher=more variable)'
        )
    ) AS AI_RECOMMENDATION

FROM tile_summary
ORDER BY
    -- Surface the most urgent tiles first
    CASE TILE_STATUS WHEN 'ALARM' THEN 1 WHEN 'WARNING' THEN 2 ELSE 3 END,
    HOTSPOT_FLAG DESC,
    AVG_TILE DESC;